# 04 — Modelagem (clusterização)

**CRISP-DM fase 4.** Segmentação de colaboradores com dados mistos.

**Entrada:** `data/processed/hr_features_cluster.csv` + tipologia  
**Saídas:** matrizes Gower/padronizada, rótulos, figuras e tabelas em `reports/`

Referência metodológica: módulos em `src/model/` (adaptados do projeto-exemplo).

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
from src.model import densidade as dens
from src.model import distancias as dist
from src.model import hierarquica as hier
from src.model import particional as part

config.garantir_diretorios()
SEMENTE = config.SEMENTE
K_MIN, K_MAX = config.K_MINIMO, config.K_MAXIMO

## 1. Entrada e fechamento da preparação

- Dissimilaridade de **Gower** (numéricas + Likert + ordinais + nominais)
- Matriz **one-hot + StandardScaler** para K-Means / DBSCAN / PCA

In [ ]:
tipologia = json.loads(
    (config.DATA_PROCESSED / "tipologia_variaveis.json").read_text(encoding="utf-8")
)
df = pd.read_csv(config.DATA_PROCESSED / "hr_features_cluster.csv")
avaliacao = pd.read_csv(config.DATA_PROCESSED / "hr_avaliacao.csv")

numericas = tipologia["numericas"] + tipologia["likert_clima"] + tipologia["ordinais"]
nominais = tipologia["categoricas_nominais"]

print(df.shape)
print("numéricas/ordinais/Likert:", len(numericas))
print("nominais:", len(nominais))

In [ ]:
gower = dist.gower(df, numericas=numericas, nominais=nominais)
np.save(config.DATA_PROCESSED / "matriz_gower.npy", gower)

df_oh = pd.get_dummies(df, columns=nominais, drop_first=False)
scaler = StandardScaler()
X = scaler.fit_transform(df_oh.to_numpy(dtype=float))
np.save(config.DATA_PROCESSED / "matriz_kmeans_scaled.npy", X)
pd.DataFrame(X, columns=df_oh.columns).to_csv(
    config.DATA_PROCESSED / "hr_kmeans_scaled.csv", index=False
)

print("Gower:", gower.shape, "min/max:", float(gower.min()), float(gower.max()))
print("Matriz padronizada:", X.shape)

## 2. Critérios de aceite (antes de rodar)

| Critério | Valor |
|---|---|
| k | 2 a 5 (restrição da diretoria) |
| Silhueta | acompanhar; base sintética pode ser fraca |
| Menor grupo | preferencialmente ≥ ~5% da base |
| Interpretação RH | obrigatória para a escolha final |

## 3. Varredura de k — K-Medoids (Gower) e K-Means

In [ ]:
var_gower = part.varrer_k_gower(gower, K_MIN, K_MAX, reinicios=8, semente=SEMENTE)
var_kmeans = part.varrer_k(X, K_MIN, K_MAX, semente=SEMENTE)

var_gower.to_csv(config.TABLES / "varredura_kmedoids_gower.csv")
var_kmeans.to_csv(config.TABLES / "varredura_kmeans.csv")

display(var_gower.round(4))
display(var_kmeans.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(var_gower.index, var_gower["silhueta"], marker="o")
axes[0].set_title("Silhueta — K-Medoids (Gower)")
axes[0].set_xlabel("k")
axes[1].plot(var_kmeans.index, var_kmeans["silhueta"], marker="o", color="#EE6C4D")
axes[1].set_title("Silhueta — K-Means (padronizado)")
axes[1].set_xlabel("k")
plt.tight_layout()
fig.savefig(config.FIGURES / "07_escolha_k.png", dpi=120, bbox_inches="tight")
fig.savefig(config.DOCS_FIGURES / "07_escolha_k.png", dpi=120, bbox_inches="tight")
plt.show()

K = int(var_gower["silhueta"].idxmax())
print("k escolhido (máx. silhueta Gower):", K)

## 4. Modelos no k escolhido

In [ ]:
res_pam = part.kmedoids(gower, k=K, reinicios=12, semente=SEMENTE)
rotulos_pam = res_pam.rotulos
print("K-Medoids custo:", round(res_pam.custo, 2))
print(pd.Series(rotulos_pam).value_counts().sort_index())

display(part.silhueta_por_grupo(None, rotulos_pam, distancias=gower))

In [ ]:
ligacoes = hier.comparar_ligacoes_gower(gower, K)
display(ligacoes)
ligacao = ligacoes["cofenetico"].idxmax()
matriz_lig = hier.matriz_ligacao(distancias=gower, ligacao=ligacao)
rotulos_hier = hier.cortar(matriz_lig, K)

fig, ax = plt.subplots(figsize=(10, 5))
hier.desenhar_dendrograma(ax, matriz_lig, K, f"Dendrograma Gower ({ligacao})")
plt.tight_layout()
fig.savefig(config.FIGURES / "08_dendrograma_gower.png", dpi=120, bbox_inches="tight")
fig.savefig(config.DOCS_FIGURES / "08_dendrograma_gower.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
rotulos_km = KMeans(n_clusters=K, n_init=20, random_state=SEMENTE).fit_predict(X)

min_amostras = max(2 * X.shape[1], 10)
curva = dens.curva_k_distancia(X, min_amostras)
_, eps = dens.joelho_k_distancia(curva)
rotulos_db = dens.aplicar_dbscan(X, eps=eps, min_amostras=min_amostras)

print(f"DBSCAN eps={eps:.3f}, min_amostras={min_amostras}")
print("grupos (sem ruído):", len(set(rotulos_db[rotulos_db >= 0])))
print("% ruído:", round((rotulos_db < 0).mean() * 100, 2))

## 5. Visualização PCA (espaço padronizado) e perfis de negócio

In [ ]:
pca = PCA(n_components=2, random_state=SEMENTE)
proj = pca.fit_transform(X)
print("Variância explicada (2D):", round(float(pca.explained_variance_ratio_.sum()), 3))

fig, ax = plt.subplots(figsize=(7, 5))
for g in sorted(np.unique(rotulos_pam)):
    m = rotulos_pam == g
    ax.scatter(proj[m, 0], proj[m, 1], s=12, alpha=0.7, label=f"Grupo {g}")
ax.set_title("K-Medoids (Gower) em PCA-2D")
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(config.FIGURES / "09_pca_clusters.png", dpi=120, bbox_inches="tight")
fig.savefig(config.DOCS_FIGURES / "09_pca_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
perfil = df.copy()
perfil["cluster"] = rotulos_pam
perfil["Attrition"] = avaliacao["Attrition"].values

resumo = (
    perfil.groupby("cluster")
    .agg(
        n=("cluster", "size"),
        pct_attrition=("Attrition", lambda s: (s == "Yes").mean() * 100),
        pct_overtime=("OverTime", lambda s: (s == "Yes").mean() * 100),
        job_satisfaction=("JobSatisfaction", "mean"),
        monthly_income=("MonthlyIncome", "median"),
        age=("Age", "median"),
        years_company=("YearsAtCompany", "median"),
    )
    .round(2)
)
resumo.to_csv(config.TABLES / "perfil_clusters_kmedoids.csv")
resumo

## 5.1 Comparativo GMM (espaço padronizado)

Varredura k=2…5 × tipos de covariância (BIC/AIC). GMM **não** substitui K-Medoids/Gower: premissa gaussiana é frágil em Likert e nominais one-hot.


In [ ]:
from src.model import mistura as mix

var_gmm = mix.varrer_gmm(X, k_minimo=K_MIN, k_maximo=K_MAX)
var_gmm.to_csv(config.TABLES / "varredura_gmm.csv", index=False)
escolha_gmm = mix.escolher_configuracao(var_gmm, k_alvo=int(K))
_, rotulos_gmm, _ = mix.ajustar_gmm(X, k=int(K), covariancia=str(escolha_gmm["covariancia"]))
(config.MODELS / "decisao_gmm.json").write_text(
    json.dumps(
        {
            "k": int(K),
            "covariancia": str(escolha_gmm["covariancia"]),
            "bic": float(escolha_gmm["bic"]),
            "silhueta": float(escolha_gmm["silhueta"]),
            "pct_fronteira": float(escolha_gmm["pct_fronteira"]),
            "papel": "comparativo no espaco one-hot + StandardScaler; nao substitui Gower",
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
print("GMM escolhido (k oficial):", dict(escolha_gmm))
var_gmm

## 6. Persistência dos rótulos e decisão

In [ ]:
rotulos_df = pd.DataFrame(
    {
        "EmployeeNumber": avaliacao["EmployeeNumber"],
        "cluster_kmedoids_gower": rotulos_pam,
        "cluster_hierarquico_gower": rotulos_hier,
        "cluster_kmeans": rotulos_km,
        "cluster_dbscan": rotulos_db,
        "cluster_gmm": rotulos_gmm,
    }
)
rotulos_df.to_csv(config.DATA_PROCESSED / "rotulos_clusters.csv", index=False)

decisao = {
    "k": int(K),
    "metodo_principal": "kmedoids_gower",
    "silhueta_gower": float(var_gower.loc[K, "silhueta"]),
    "ligacao_hierarquica": str(ligacao),
    "dbscan_eps": float(eps),
    "dbscan_min_amostras": int(min_amostras),
    "dbscan_pct_ruido": float((rotulos_db < 0).mean()),
    "observacao": (
        "Silhueta baixa é compatível com o enunciado do Tema 08 "
        "(estrutura de grupos artificialmente fraca na base sintética). "
        "A escolha privilegia interpretabilidade de RH."
    ),
}
(config.MODELS / "decisao_modelagem.json").write_text(
    json.dumps(decisao, indent=2, ensure_ascii=False), encoding="utf-8"
)
decisao

### Leitura de negócio (k=2, K-Medoids/Gower)

Com base no perfil mediano típico desta execução:

- **Grupo de maior estabilidade:** renda e tempo de casa mais altos, attrition menor.
- **Grupo de maior risco:** renda e tempo de casa menores, attrition e overtime mais altos.

Próxima fase (avaliação): confrontar hipóteses do Canvas, medir estabilidade entre métodos e detalhar ações por perfil.